In [5]:
import pandas as pd
import ast
from alive_progress import alive_bar

In [6]:
# Load and analyze data
edges_df = pd.read_csv('../output/scored_edges_saigon.csv')

In [7]:
new_df = []  # Temporary list to hold extracted metric rows
columns = ['abs_max_slope', 'abs_elevation_diff', 'max_abs_elevation_diff']  # Desired output columns

# Loop through each row in the original DataFrame
for index, row in edges_df.iterrows():
    # Convert the string representation of the dictionary back into a real dictionary
    metrics_dict = ast.literal_eval(row["slope_metrics"])
    
    # Extract relevant values in the correct order and append to the list
    new_df.append([
        metrics_dict['abs_max_slope'],
        metrics_dict['abs_elevation_diff'],
        metrics_dict['max_abs_elevation_diff']
    ])

# Convert the list of lists into a new DataFrame with proper column names
new_df = pd.DataFrame(new_df, columns=columns)

# Preview the data and verify everything looks correct
print(new_df.head())       # First few rows of extracted metrics
print(new_df.info())       # Summary info about data types and missing values
print(new_df.describe())   # Statistical summary of the numeric columns


   abs_max_slope  abs_elevation_diff  max_abs_elevation_diff
0       0.003543            0.046227                0.046227
1       0.008239            0.102371                0.102371
2       0.000648            0.005920                0.005920
3       0.085021            0.816582                0.816582
4       0.046776            0.491718                0.491718
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2456 entries, 0 to 2455
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   abs_max_slope           2456 non-null   float64
 1   abs_elevation_diff      2456 non-null   float64
 2   max_abs_elevation_diff  2456 non-null   float64
dtypes: float64(3)
memory usage: 57.7 KB
None
       abs_max_slope  abs_elevation_diff  max_abs_elevation_diff
count    2456.000000         2456.000000             2456.000000
mean        0.052935            1.143515                1.176454
std         0.056068  

In [ ]:
accesscores_slope = []  # Final adjusted access scores (accounting for slope & elevation)
notes = []              # Store notes about slope and elevation for each segment

with alive_bar(len(edges_df), force_tty=True) as bar:
    try:
        for count in range(len(edges_df)):
                # Slope penalty term:
            max_slope = new_df.iloc[count]['abs_max_slope']
            p1 = (max_slope - 0.083) * (max_slope + 1) if max_slope > 0.083 else 0
            p2 = new_df.iloc[count]['max_abs_elevation_diff'] / 100
            adjusted_score = edges_df.iloc[count]['AccessScore'] - p1 - p2
            accesscores_slope.append(max(adjusted_score, 0))
            notes.append({
                "max_slope": new_df.iloc[count]['abs_max_slope'],
                "elevation_change": new_df.iloc[count]['abs_elevation_diff']
            })
            bar()
    except Exception as e:
        print("Partial scores:", accesscores_slope)
        raise e

|████████████████████████████████████████| 2456/2456 [100%] in 0.7s (3399.97/s) 


In [9]:
edges_df['AccessScore_slope'] = accesscores_slope
edges_df['slope_notes'] = notes
edges_df.to_csv('../output/scored_edges_saigon.csv', index=False)